In [1]:
import pandas as pd
pokemon = pd.read_csv(r'C:\Users\akals\Downloads\Pokemon.csv')

In [2]:
pokemon['name_count'] = pokemon['Name'].str.len()
pokemon['long_name'] = pokemon['name_count'] >= 10

In [3]:
pokemon['name_nospace'] = pokemon['Name'].str.replace(' ', '')

In [4]:
pokemon['name_isalpha'] = pokemon['name_nospace'].str.isalpha()

In [5]:
pokemon[pokemon['name_isalpha'] == False]['Name']

34             Nidoran♀
37             Nidoran♂
90           Farfetch'd
131            Mr. Mime
252            Porygon2
270               Ho-oh
487            Mime Jr.
525           Porygon-Z
794    Zygarde50% Forme
Name: Name, dtype: object

pokemon['Name'] = pokemon['Name'].replace({
    'Nidoran♀': 'NidoranF',
    'Nidoran♂': 'NidoranM',
    "Farfetch'd": 'Farfetchd',
    'Mr. Mime': 'MrMime',
    'Porygon2': 'PorygonTwo',
    'Ho-oh': 'Hooh',
    'Mime Jr.': 'MimeJr',
    'Porygon-Z': 'PorygonZ',
    'Zygarde50% Forme': 'Zygarde Forme'
})

import re

In [6]:
import re

In [7]:
name = 'CharizardMega Charizard X'
print(name.split(' '))

['CharizardMega', 'Charizard', 'X']


In [8]:
temp = 'CharizardMega'
print(re.findall('[A-Z][a-z]*', temp))

['Charizard', 'Mega']


In [9]:
def tokenize(name):
    tokens = []
    name_split = name.split(' ')
    for part in name_split:
        token_list = re.findall('[A-Z][a-z]*',part)
        tokens.extend(token_list)
    return tokens

In [10]:
legendary_pokemon = pokemon[pokemon['Legendary']==True]

token_list = []
for name in legendary_pokemon['Name']:
    token_list.extend(tokenize(name))

print(len(set(token_list)))

65


In [11]:
from collections import Counter
token_counter = Counter(token_list)
print(token_counter.most_common(10))

[('Forme', 15), ('Mega', 6), ('Mewtwo', 5), ('Kyurem', 5), ('Deoxys', 4), ('Hoopa', 4), ('Latias', 3), ('Latios', 3), ('Kyogre', 3), ('Groudon', 3)]


In [12]:
pokemon['has_Forme'] = pokemon['Name'].str.contains('Forme')
pokemon['has_Mega'] = pokemon['Name'].str.contains('Mega')

In [13]:
pokemon[['Name', 'has_Forme', 'has_Mega']].head(10)

,Name,has_Forme,has_Mega
0,Bulbasaur,False,False
1,Ivysaur,False,False
2,Venusaur,False,False
3,VenusaurMega Venusaur,False,True
4,Charmander,False,False
5,Charmeleon,False,False
6,Charizard,False,False
7,CharizardMega Charizard X,False,True
8,CharizardMega Charizard Y,False,True
9,Squirtle,False,False


In [14]:
pokemon = pd.get_dummies(pokemon, columns=['Type 1', 'Type 2'])
pokemon.head()

,#,Name,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,...,Type 2_Ghost,Type 2_Grass,Type 2_Ground,Type 2_Ice,Type 2_Normal,Type 2_Poison,Type 2_Psychic,Type 2_Rock,Type 2_Steel,Type 2_Water
0,1,Bulbasaur,318,45,49,49,65,65,45,1,...,False,False,False,False,False,True,False,False,False,False
1,2,Ivysaur,405,60,62,63,80,80,60,1,...,False,False,False,False,False,True,False,False,False,False
2,3,Venusaur,525,80,82,83,100,100,80,1,...,False,False,False,False,False,True,False,False,False,False
3,3,VenusaurMega Venusaur,625,80,100,123,122,120,80,1,...,False,False,False,False,False,True,False,False,False,False
4,4,Charmander,309,39,52,43,60,50,65,1,...,False,False,False,False,False,False,False,False,False,False


In [17]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

features2 = pokemon.columns.tolist()
remove_cols = ['Name', 'Legendary', 'name_nospace', 'name_isalpha']
features2 = [col for col in features2 if col not in remove_cols]

X2 = pokemon[features2]
y2 = pokemon['Legendary']
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)

In [18]:
from sklearn.ensemble import RandomForestClassifier

model3 = RandomForestClassifier(n_estimators=100, random_state=42)
model3.fit(X2_train, y2_train)

y3_pred = model3.predict(X2_test)
print('Accuracy:', accuracy_score(y2_test, y3_pred))
print('F1 Score:', f1_score(y2_test, y3_pred))

Accuracy: 0.975
F1 Score: 0.8181818181818182


In [19]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 20, None],
    'min_samples_leaf': [1, 2, 5]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    scoring='f1',
    cv=5
)
grid_search.fit(X2_train, y2_train)

print('최적 파라미터:', grid_search.best_params_)
print('최적 F1:', grid_search.best_score_)

최적 파라미터: {'max_depth': 10, 'min_samples_leaf': 1, 'n_estimators': 100}
최적 F1: 0.7924386724386725


In [23]:
best_model = grid_search.best_estimator_
y_best_pred = best_model.predict(X2_test)
print('Accuracy:', accuracy_score(y2_test, y_best_pred))
print('F1 Score:', f1_score(y2_test, y_best_pred))

Accuracy: 0.975
F1 Score: 0.8181818181818182


In [24]:
model4 = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42)
model4.fit(X2_train, y2_train)

y4_pred = model4.predict(X2_test)
print('Accuracy:', accuracy_score(y2_test, y4_pred))
print('F1 Score:', f1_score(y2_test, y4_pred))

Accuracy: 0.9625
F1 Score: 0.7272727272727273


In [25]:
y_proba = model3.predict_proba(X2_test)[:, 1]
print(y_proba[:10])

[0.76 0.05 0.   0.01 0.   0.05 0.01 0.02 0.   0.04]


In [26]:
from sklearn.metrics import precision_score, recall_score

for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    y_custom = y_proba >= threshold
    p = precision_score(y2_test, y_custom)
    r = recall_score(y2_test, y_custom)
    f = f1_score(y2_test, y_custom)
    print(f'기준선 {threshold}: Precision={p:.3f}, Recall={r:.3f}, F1={f:.3f}')

기준선 0.3: Precision=0.714, Recall=1.000, F1=0.833
기준선 0.4: Precision=0.714, Recall=1.000, F1=0.833
기준선 0.5: Precision=0.750, Recall=0.900, F1=0.818
기준선 0.6: Precision=0.700, Recall=0.700, F1=0.700
기준선 0.7: Precision=0.857, Recall=0.600, F1=0.706


In [27]:
import pandas as pd

importance = pd.Series(model3.feature_importances_, index=X2.columns)
print(importance.sort_values(ascending=False).head(15))

Total             0.230066
Speed             0.101930
Sp. Atk           0.087750
Sp. Def           0.086010
HP                0.083054
#                 0.067579
Attack            0.056008
Defense           0.051186
name_count        0.045125
has_Forme         0.029182
Generation        0.022521
has_Mega          0.019979
long_name         0.013350
Type 1_Psychic    0.011565
Type 1_Water      0.009248
dtype: float64


In [28]:
top_features = ['Total', 'Speed', 'Sp. Atk', 'Sp. Def', 'HP', 'Attack', 'Defense', '#']

X3 = pokemon[top_features]
X3_train, X3_test, y3_train, y3_test = train_test_split(X3, y2, test_size=0.2, random_state=42)

model5 = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
model5.fit(X3_train, y3_train)

y5_proba = model5.predict_proba(X3_test)[:, 1]
y5_custom = y5_proba >= 0.3
print('F1 Score:', f1_score(y3_test, y5_custom))

F1 Score: 0.7692307692307693


In [31]:
pokemon['atk_ratio'] = pokemon['Sp. Atk'] / (pokemon['Attack'] + 1)
pokemon['def_total'] = pokemon['Defense'] + pokemon['Sp. Def']
pokemon['speed_hp'] = pokemon['Speed'] / (pokemon['HP'] + 1)

In [32]:
features_new = features2 + ['atk_ratio', 'def_total', 'speed_hp']
X4 = pokemon[features_new]
X4_train, X4_test, y4_train, y4_test = train_test_split(X4, y2, test_size=0.2, random_state=42)

model6 = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
model6.fit(X4_train, y4_train)

y6_proba = model6.predict_proba(X4_test)[:, 1]
y6_custom = y6_proba >= 0.3
print('F1 Score:', f1_score(y4_test, y6_custom))

F1 Score: 0.8333333333333334


In [33]:
!pip install imbalanced-learn


Defaulting to user installation because normal site-packages is not writeable

   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   ---------------------------------------- 2/2 [imbalanced-learn]



In [34]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X2_train_sm, y2_train_sm = smote.fit_resample(X2_train, y2_train)

print('원래 학습 데이터:', y2_train.value_counts().to_dict())
print('SMOTE 후:', y2_train_sm.value_counts().to_dict())

원래 학습 데이터: {False: 585, True: 55}
SMOTE 후: {True: 585, False: 585}


In [35]:
model7 = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
model7.fit(X2_train_sm, y2_train_sm)

y7_proba = model7.predict_proba(X2_test)[:, 1]
y7_custom = y7_proba >= 0.3

print('F1 Score:', f1_score(y2_test, y7_custom))
print('Precision:', precision_score(y2_test, y7_custom))
print('Recall:', recall_score(y2_test, y7_custom))

F1 Score: 0.8
Precision: 0.6666666666666666
Recall: 1.0


In [36]:
for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    y_custom = y7_proba >= threshold
    p = precision_score(y2_test, y_custom)
    r = recall_score(y2_test, y_custom)
    f = f1_score(y2_test, y_custom)
    print(f'기준선 {threshold}: Precision={p:.3f}, Recall={r:.3f}, F1={f:.3f}')

기준선 0.3: Precision=0.667, Recall=1.000, F1=0.800
기준선 0.4: Precision=0.667, Recall=1.000, F1=0.800
기준선 0.5: Precision=0.692, Recall=0.900, F1=0.783
기준선 0.6: Precision=0.636, Recall=0.700, F1=0.667
기준선 0.7: Precision=0.700, Recall=0.700, F1=0.700


In [37]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_leaf': [1, 2, 3, 5],
    'min_samples_split': [2, 5, 10]
}

grid_search2 = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    scoring='f1',
    cv=5
)
grid_search2.fit(X2_train_sm, y2_train_sm)

print('최적 파라미터:', grid_search2.best_params_)
print('최적 F1 (CV):', grid_search2.best_score_)

최적 파라미터: {'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
최적 F1 (CV): 0.9865468783522848


In [38]:
best_model2 = grid_search2.best_estimator_
y_best2_proba = best_model2.predict_proba(X2_test)[:, 1]

for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    y_custom = y_best2_proba >= threshold
    p = precision_score(y2_test, y_custom)
    r = recall_score(y2_test, y_custom)
    f = f1_score(y2_test, y_custom)
    print(f'기준선 {threshold}: Precision={p:.3f}, Recall={r:.3f}, F1={f:.3f}')

기준선 0.3: Precision=0.667, Recall=1.000, F1=0.800
기준선 0.4: Precision=0.714, Recall=1.000, F1=0.833
기준선 0.5: Precision=0.692, Recall=0.900, F1=0.783
기준선 0.6: Precision=0.727, Recall=0.800, F1=0.762
기준선 0.7: Precision=0.700, Recall=0.700, F1=0.700


In [39]:
!pip install xgboost

Defaulting to user installation because normal site-packages is not writeable


In [40]:
from xgboost import XGBClassifier

model8 = XGBClassifier(n_estimators=200, max_depth=10, random_state=42, eval_metric='logloss')
model8.fit(X2_train_sm, y2_train_sm)

y8_proba = model8.predict_proba(X2_test)[:, 1]

for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    y_custom = y8_proba >= threshold
    p = precision_score(y2_test, y_custom)
    r = recall_score(y2_test, y_custom)
    f = f1_score(y2_test, y_custom)
    print(f'기준선 {threshold}: Precision={p:.3f}, Recall={r:.3f}, F1={f:.3f}')

기준선 0.3: Precision=0.769, Recall=1.000, F1=0.870
기준선 0.4: Precision=0.769, Recall=1.000, F1=0.870
기준선 0.5: Precision=0.769, Recall=1.000, F1=0.870
기준선 0.6: Precision=0.769, Recall=1.000, F1=0.870
기준선 0.7: Precision=0.769, Recall=1.000, F1=0.870


In [41]:
from sklearn.model_selection import GridSearchCV

param_grid_xgb = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.3],
    'min_child_weight': [1, 3, 5]
}

grid_xgb = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric='logloss'),
    param_grid_xgb,
    scoring='f1',
    cv=5
)
grid_xgb.fit(X2_train_sm, y2_train_sm)

print('최적 파라미터:', grid_xgb.best_params_)

최적 파라미터: {'learning_rate': 0.3, 'max_depth': 3, 'min_child_weight': 1, 'n_estimators': 200}


In [42]:
best_xgb = grid_xgb.best_estimator_
y_xgb_proba = best_xgb.predict_proba(X2_test)[:, 1]

for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    y_custom = y_xgb_proba >= threshold
    p = precision_score(y2_test, y_custom)
    r = recall_score(y2_test, y_custom)
    f = f1_score(y2_test, y_custom)
    print(f'기준선 {threshold}: Precision={p:.3f}, Recall={r:.3f}, F1={f:.3f}')

기준선 0.3: Precision=0.769, Recall=1.000, F1=0.870
기준선 0.4: Precision=0.769, Recall=1.000, F1=0.870
기준선 0.5: Precision=0.769, Recall=1.000, F1=0.870
기준선 0.6: Precision=0.769, Recall=1.000, F1=0.870
기준선 0.7: Precision=0.769, Recall=1.000, F1=0.870


In [43]:
false_positives = X2_test[(y_best2_proba >= 0.3) & (y2_test == False)]
pokemon.loc[false_positives.index]['Name']

696                  Hydreigon
275      SceptileMega Sceptile
715         MeloettaAria Forme
409    SalamenceMega Salamence
776                     Goodra
Name: Name, dtype: object

In [44]:
pokemon['is_mega'] = pokemon['Name'].str.contains('Mega')
pokemon['total_600'] = pokemon['Total'] == 600

In [45]:
features3 = features2 + ['is_mega', 'total_600']
X5 = pokemon[features3]
X5_train, X5_test, y5_train, y5_test = train_test_split(X5, y2, test_size=0.2, random_state=42)

smote2 = SMOTE(random_state=42)
X5_train_sm, y5_train_sm = smote2.fit_resample(X5_train, y5_train)

model9 = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.3, random_state=42, eval_metric='logloss')
model9.fit(X5_train_sm, y5_train_sm)

y9_proba = model9.predict_proba(X5_test)[:, 1]

for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    y_custom = y9_proba >= threshold
    p = precision_score(y2_test, y_custom)
    r = recall_score(y2_test, y_custom)
    f = f1_score(y2_test, y_custom)
    print(f'기준선 {threshold}: Precision={p:.3f}, Recall={r:.3f}, F1={f:.3f}')

기준선 0.3: Precision=0.769, Recall=1.000, F1=0.870
기준선 0.4: Precision=0.769, Recall=1.000, F1=0.870
기준선 0.5: Precision=0.769, Recall=1.000, F1=0.870
기준선 0.6: Precision=0.769, Recall=1.000, F1=0.870
기준선 0.7: Precision=0.769, Recall=1.000, F1=0.870
